# 小鼠矢状前脑与后脑切片

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import torch

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (cwd, *cwd.parents) if (path / "SpaDiff_improved").is_dir()),
    cwd,
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import SpaDiff_improved as sd
from SpaDiff_improved.spatial import Neiber
from SpaDiff_improved.utils import adjust_louvain_resolution, set_seed

In [ ]:
SEED = 42
SLICE_ORDER = ["A", "P"]
REFERENCE_BATCH = "A"
N_CLUSTERS = 30
N_COMPONENTS = 50
K_INTRA = 6
K_INTER = 2
FAST_DEV_RUN = False

# loss = K1 * original DEC-KL + K2 * diffusion DSM
K1 = 1.0
K2 = 1.0
DEC_ONLY = K1 > 0.0 and K2 == 0.0

DATA_ROOT = Path("E:/gxy_2/final/0_data/case2/")
print("DATA_ROOT =", DATA_ROOT)

set_seed(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device("cpu") if DEC_ONLY else torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)
print("device =", device)
print(f"loss = {K1} * original_DEC + {K2} * diffusion_DSM")


## 读取 Anterior/Posterior Visium 数据并对齐坐标

In [ ]:
adata_a = sc.read_visium(DATA_ROOT / "Anterior")
adata_p = sc.read_visium(DATA_ROOT / "Posterior")
adata_a.var_names_make_unique()
adata_p.var_names_make_unique()

anterior_coord = np.asarray(adata_a.obsm["spatial"]).copy()
posterior_coord = np.asarray(adata_p.obsm["spatial"]).copy()
posterior_coord[:, 1] += anterior_coord[:, 1].min() - posterior_coord[:, 1].min()
posterior_coord[:, 0] += anterior_coord[:, 0].max() - posterior_coord[:, 0].min()
adata_p.obsm["spatial"] = posterior_coord

adata = sc.concat(
    {"A": adata_a, "P": adata_p},
    join="inner", label="batch_name", index_unique="-",
)
adata.obs["batch_name"] = pd.Categorical(
    adata.obs["batch_name"], categories=SLICE_ORDER, ordered=True
)
adata.layers["counts"] = adata.X.copy()
print(adata)
print(adata.obs["batch_name"].value_counts())

## HVG、归一化与 PCA

In [ ]:
sc.pp.highly_variable_genes(
    adata, flavor="seurat_v3", layer="counts",
    n_top_genes=min(3000, adata.n_vars), batch_key="batch_name",
)
adata = adata[:, adata.var["highly_variable"]].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.scale(adata, max_value=10)
latent_dim = min(N_COMPONENTS, adata.n_obs - 1, adata.n_vars - 1)
if latent_dim < 2:
    raise ValueError("可用 spot 或基因过少，无法构建 PCA 表示")
sc.tl.pca(adata, n_comps=latent_dim, svd_solver="arpack")
features = torch.as_tensor(
    np.asarray(adata.obsm["X_pca"], dtype=np.float32), device=device
)
print("PCA feature shape =", tuple(features.shape))

## 构建单纯复形

In [ ]:
_, adjacency = Neiber(
    adata, k_intra=K_INTRA, k_inter=K_INTER, slice_order=SLICE_ORDER
)
adjacency = adjacency.maximum(adjacency.T)
operators = sd.to_torch_operators(
    sd.build_simplicial_operators(adjacency, max_order=2), device=device
)
batch_codes = adata.obs["batch_name"].cat.codes.to_numpy()

batch_ids = torch.as_tensor(batch_codes, dtype=torch.long, device=device)
modality_ids = torch.zeros(adata.n_obs, dtype=torch.long, device=device)
print("edge nnz =", operators[1]._nnz())
print("triangle nnz =", operators[2]._nnz())

## 训练批次条件 VP-SDE

In [ ]:
config = sd.SpaDiffConfig(
    data_dim=latent_dim,
    condition_input_dim=latent_dim,
    hidden_dim=128,
    topology_hidden_dim=64,
    topology_dim=64,
    score_depth=4,
    dropout=0.2,
    topology_projection_dropout=0.1 if K1 > 0.0 else None,
    simplex_orders=(1, 2),
    propagation_steps=5,
    propagation_alpha=0.4,
    num_batches=len(SLICE_ORDER),
    num_modalities=1,
    num_scales=51000,
    k1=K1,
    k2=K2,
    num_clusters=N_CLUSTERS,
    dec_alpha=1.0,
    dec_update_interval=10,
    dec_tolerance=1e-3,
    random_seed=SEED,
)
model = sd.SpaDiff(config).to(device)

training_epochs = 500
training = sd.train_spadiff(
    model,
    target_features=features,
    operators=operators,
    batch_ids=batch_ids,
    modality_ids=modality_ids,
    condition_features=features,
    epochs=training_epochs,
    learning_rate=1e-3,
    weight_decay=1e-4,
    ema_decay=None if DEC_ONLY else 0.999,
    verbose_every=1 if FAST_DEV_RUN or DEC_ONLY else 25,
)
print("best total loss =", training.best_loss)
print("last original DEC loss =", training.original_losses[-1])
print("last diffusion loss =", training.diffusion_losses[-1])


## 使用EMA权重向参考切片条件去噪


In [ ]:
if DEC_ONLY:
    dec_labels, dec_prob, embedding = model.original_predict(features, operators)
    corrected = embedding
else:
    reference_code = SLICE_ORDER.index(REFERENCE_BATCH)
    reference_batch_ids = torch.full_like(batch_ids, reference_code)
    if training.ema is not None:
        training.ema.store(model.parameters())
        training.ema.copy_to(model.parameters())
    try:
        corrected = model.harmonize(
            observed_features=features,
            operators=operators,
            reference_batch_ids=reference_batch_ids,
            modality_ids=modality_ids,
            condition_features=features,
            strength=0.35,
            sampler="ode",
            guidance_scale=1.1,
            ode_steps=20 if FAST_DEV_RUN else 250,
        )
    finally:
        if training.ema is not None:
            training.ema.restore(model.parameters())

adata.obsm["X_spadiff"] = corrected.cpu().numpy()
print("SpaDiff embedding shape =", adata.obsm["X_spadiff"].shape)


## Louvain聚类与聚类结果可视化


In [ ]:
adata = adjust_louvain_resolution(
    adata,
    target_n_clusters=N_CLUSTERS,
    use_rep="X_spadiff",
    key_added="louvain",
    n_neighbors=15,
    random_state=SEED,
    resolution_bounds=(0.01, 5.0),
    tolerance=0,
    max_iterations=25,
    verbose=True,
)
search = adata.uns["louvain_resolution_search"]
print("selected resolution =", search["selected_resolution"])
print("selected clusters =", search["selected_n_clusters"])

In [ ]:
plot_color = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
    "#f2a6d4", "#6a9c0a", "#d1581e", "#3b4e97", "#66b200",
    "#ff5c8f", "#ffcc00", "#ff8d4d", "#62c4da", "#a7a7a7",
    "#e1c39b", "#9c6c6c", "#c13b5b", "#5c82b3", "#ba8dff",
    "#b8c239", "#f2b1d9", "#2c6d8f", "#ff6f61", "#4daf4a",
]
fig, axes = plt.subplots(1, len(SLICE_ORDER), figsize=(12, 5))
for axis, sample in zip(np.atleast_1d(axes), SLICE_ORDER):
    subset = adata[adata.obs["batch_name"] == sample].copy()
    sc.pl.spatial(
        subset, img_key=None, color="louvain", ax=axis, show=False,
        spot_size=120, palette=plot_color, legend_loc=None,
        frameon=False, title=sample,
    )
plt.tight_layout()
plt.show()

In [ ]:

sc.tl.umap(adata, random_state=SEED)
sc.pl.umap(
    adata, color=["batch_name", "louvain"], size=10,
    legend_fontsize=11, legend_fontoutline=2,
)